# Explicação do `run_full_factorial.sh`

Para automatizar o processo de geração de dados foi criado o script `run_full_factorial.sh`, que realiza estas funções:

1. Compilação de gerador.cpp e sorting.cpp;
2. Teste full factorial;
3. 5 repetições do teste;

Todo script foi gerado utilizando IA do ChatGPT para codificá-lo.

# Estrutura esperada do projeto

```text
src/
│
├── generador.cpp
├── sorting.cpp
├── run_full_factorial.sh
├── pre_run_setup.sh
├── datasets/
└── results/
```
Caso o run_full_factorial.sh ou pre_run_setup.sh não esteja na pasta com os arquivos .cpp, não funcionará

# Código completo do `run_full_factorial.sh`

In [ ]:
#!/usr/bin/env bash

set -euo pipefail

SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" && pwd)"
cd "$SCRIPT_DIR"

GERADOR_EXE="$SCRIPT_DIR/gerador"
SORTING_EXE="$SCRIPT_DIR/sorting"
DATASETS_DIR="$SCRIPT_DIR/datasets"
RESULTS_DIR="$SCRIPT_DIR/results"

mkdir -p "$DATASETS_DIR"
mkdir -p "$RESULTS_DIR"

echo "Compilando gerador..."
g++ gerador.cpp -std=c++17 -O2 -o "$GERADOR_EXE"

echo "Compilando sorting..."
g++ sorting.cpp -std=c++17 -O2 -pthread -o "$SORTING_EXE"

ALGORITHMS=(heap merge quick)
STRESSES=(none cpu ram both)
SIZES=(7)
REPS=(1 2 3 4 5)
CARGAS=(0 0.5 1.0)

for carga in "${CARGAS[@]}"; do
  echo "### Carga $carga ###"

  for rep in "${REPS[@]}"; do
    echo "=== Repetição $rep / ${#REPS[@]} ==="

    for size in "${SIZES[@]}"; do
      echo "Gerando datasets para tamanho 10^$size..."
      "$GERADOR_EXE" "$size" 10

      mapfile -t dataset_files < <(find "$DATASETS_DIR" -maxdepth 1 -type f -name '*.bin' | sort)

      for dataset in "${dataset_files[@]}"; do
        dataset_name="$(basename "$dataset")"
        for algorithm in "${ALGORITHMS[@]}"; do
          for stress in "${STRESSES[@]}"; do
            # Carga 0 -> só roda "none" (sem estresse nenhum).
            # Carga > 0 -> roda só cpu/ram/both, "none" já foi coberto na carga 0.
            if [[ "$carga" == "0" ]]; then
              if [[ "$stress" != "none" ]]; then
                continue
              fi
            else
              if [[ "$stress" == "none" ]]; then
                continue
              fi
            fi

            echo "Carga $carga | Rep $rep | Size 10^$size | Dataset $dataset_name | Alg $algorithm | Stress $stress"

            if [[ "$carga" == "0" ]]; then
              "$SORTING_EXE" "$algorithm" "$dataset" "$stress"
            else
              "$SORTING_EXE" "$algorithm" "$dataset" "$stress" "$carga"
            fi
          done
        done
      done
    done

  done

done

echo "Full factorial concluído. Resultados em: $RESULTS_DIR"

# Explicação do Script

O script utiliza uma seed base (SEED = 10) para garantir a reprodutibilidade dos experimentos, assegurando que os mesmos vetores sejam gerados em execuções futuras.

O funcionamento consiste em um processo automatizado de benchmark. Para cada repetição, o script executa o gerador de datasets, produzindo arquivos .bin dentro da pasta datasets/.

Em seguida, o script percorre todos os arquivos dessa pasta e executa os três algoritmos de ordenação (heap, merge e quick), combinando cada um deles com diferentes configurações de estresse de sistema. Cada execução utiliza um dataset específico como entrada.

Os resultados de cada execução são registrados em um arquivo results.csv dentro da pasta results/, permitindo posterior análise estatística e comparação de desempenho entre algoritmos e cenários.

Durante a execução, o script exibe mensagens de progresso, informando o estado atual do experimento (repetição, tamanho, dataset, algoritmo e configuração de estresse). Isso facilita o acompanhamento do processo e ajuda a identificar possíveis gargalos ou etapas ainda pendentes.

# Tamanho da entrada

Foi escolhido que todos vetores terão tamanho de 10^7 (10 milhões de instâncias), pois o grupo considerou que é o ponto ideal entre vetores longos suficientes para que o tempo de execução seja menos afetado por fatores randômicos e incontroláveis do hardware e vetores curtos suficientes para o tempo de benchmark ser acessível


# Execuções
8 vetores * 3 algoritmos * 7 níveis de estresse = 168 tipos

96 tipos * 5 repetições = 840 execuções

# Estrutura Experimental

O benchmark segue uma execução em full factorial com repetição, combinando diferentes tamanhos de entrada, datasets, algoritmos de ordenação e cenários de estresse do sistema.

A estrutura geral pode ser descrita como:

```text
para cada carga:
    para cada repetição:
        para cada tamanho:
            gerar datasets

            para cada dataset:
                para cada algoritmo:
                    para cada cenário de estresse:
                        se (carga == 0 e cenário != none):
                            continuar
                        se (carga > 0 e cenário == none):
                            continuar

                        executar benchmark
                        salvar resultado
```

Cada execução do benchmark é realizada como um processo separado, o que permite:

isolamento de memória entre execuções;

medição mais precisa do pico de uso de RAM;

redução de interferência entre algoritmos e cenários de stress;

maior estabilidade na coleta de métricas de desempenho e energia.


# Código completo do `pre_run_setup.sh`

#!/usr/bin/env bash

set -e

clear

chmod +x run_full_factorial.sh

sudo chmod -R a+r /sys/class/powercap/intel-rapl

rm -f results/*

systemd-inhibit \
    --what=idle:sleep:shutdown \
    --why="Rodando benchmark" \
    ./run_full_factorial.sh

# Explicação do Script
O script pre_run_setup.sh é responsável por preparar o ambiente antes da execução dos experimentos, garantindo que todas as condições necessárias para a coleta dos benchmarks sejam atendidas.

Inicialmente, o script limpa o terminal e garante que o arquivo run_full_factorial.sh possua permissão de execução. Em seguida, concede permissão de leitura aos arquivos da interface Intel RAPL (/sys/class/powercap/intel-rapl), permitindo que o benchmark registre o consumo de energia durante as execuções.

Após essa etapa, o script remove todos os resultados anteriormente armazenados na pasta results/, evitando que dados de execuções passadas sejam misturados aos novos resultados.

Por fim, o script inicia o benchmark utilizando o comando systemd-inhibit. Durante toda a execução dos experimentos, esse comando impede que o sistema entre em suspensão, desligue automaticamente ou fique ocioso, evitando interrupções que poderiam comprometer a execução completa dos testes ou introduzir variabilidade nos resultados experimentais.

Dessa forma, o pre_run_setup.sh automatiza toda a preparação do ambiente de execução, reduzindo a possibilidade de erros operacionais e garantindo que todos os experimentos sejam realizados sob as mesmas condições iniciais.

# Como rodar Testes Automáticos

In [ ]:
chmod +x pre_run_setup.sh
./pre_run_setup.sh